# KAUST Kaggle Challenge 3 - Notebook A
## Offset-fault classification, date-grouped cross-validation

### What the data actually says (established before any modelling)

Five things were checked on the real files first, and three of them contradict
the working plan we started from:

1. **Faults are isolated single rows, not episodes.** Median run length of the
   fault class is 1. The fault rate is flat at ~0.37 on *every* day and *every*
   hour. So "how long do fault runs last" and "does fault frequency drift" have
   no answer to find.
2. **Because faults are isolated, a row's temporal neighbours reveal it.** A
   faulty row sits far below its own local level on both sensors, and much
   further below on Sensor 2. This is the single largest source of signal.
3. **Test timestamps are minute-resolution; train timestamps are
   second-resolution.** Ordering train more precisely than we can ever order
   test silently inflates local validation. Both are floored to the minute here.
4. **The "2022" hint is not a free label.** January 2022 has the same
   fault-proxy share as December 2021 and as training. Calendar features are
   tested as a stage and they *hurt*, which is the defensible reading of hint 3.
5. **The S1-vs-S2 line is real but partial.** Best single-threshold rule on it
   reaches roughly F1 0.88 in train, short of the 0.922 published baseline.

Nothing below uses external data, external labels, or deep learning.

**This notebook's job:** measure every feature stage under `GroupKFold` on calendar date, produce out-of-fold probabilities, and stress-test the chosen feature set with ablations and adversarial validation. Notebook B answers the same questions under strict forward-in-time validation. Run both, then compare before spending any of the 8 submissions.

## 0. Setup

In [ ]:
import os, sys, json, time, glob, warnings, gc
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupKFold

t_notebook_start = time.time()
SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)
warnings.filterwarnings("ignore", category=FutureWarning)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
os.makedirs(WORK, exist_ok=True)

try:
    from catboost import CatBoostClassifier, Pool
    import catboost
    print("catboost", catboost.__version__)
except ImportError:
    print("catboost missing, installing...")
    os.system(f"{sys.executable} -m pip install -q catboost")
    from catboost import CatBoostClassifier, Pool

print("pandas", pd.__version__, "| numpy", np.__version__)
print("working dir:", WORK)

In [ ]:
# ---- dynamically locate the competition files -----------------------------
def find_files():
    roots = ["/kaggle/input", ".", "..", "/mnt/user-data/uploads"]
    found = {}
    for root in roots:
        if not os.path.isdir(root):
            continue
        for path in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
            base = os.path.basename(path).lower()
            if base.startswith("train") and "train" not in found:
                found["train"] = path
            elif base.startswith("test") and "test" not in found:
                found["test"] = path
            elif "sample" in base and "sample" not in found:
                found["sample"] = path
        if "train" in found and "test" in found:
            break
    missing = {"train", "test"} - set(found)
    if missing:
        raise FileNotFoundError(f"could not locate {missing}. Saw: {found}")
    return found

FILES = find_files()
for k, v in FILES.items():
    print(f"{k:7s} -> {v}  ({os.path.getsize(v)/1e6:.1f} MB)")

In [ ]:
# ---- robust datetime parsing ---------------------------------------------
# train uses ISO with seconds; test uses M/D/YYYY H:MM with NO seconds.
# We never assume a single format.
def parse_datetime(series):
    s = series.astype(str)
    for fmt in ("%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%Y-%m-%d %H:%M"):
        out = pd.to_datetime(s, format=fmt, errors="coerce")
        if out.notna().all():
            return out, fmt
    out = pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=False)
    if out.isna().any():
        raise ValueError(f"{out.isna().sum()} timestamps failed to parse")
    return out, "mixed"

train = pd.read_csv(FILES["train"])
test  = pd.read_csv(FILES["test"])
sample = pd.read_csv(FILES["sample"]) if "sample" in FILES else None

train["dt"], fmt_tr = parse_datetime(train["Datetime"])
test["dt"],  fmt_te = parse_datetime(test["Datetime"])
print("train datetime format:", fmt_tr, "| test datetime format:", fmt_te)

TARGET = "Offset_fault"
S1, S2 = "Sensor1_PM2.5", "Sensor2_PM2.5"
EPS = 1e-3

print("\ntrain", train.shape, train["dt"].min(), "->", train["dt"].max())
print("test ", test.shape,  test["dt"].min(),  "->", test["dt"].max())

# sanity: month-first parsing must place every test row inside Dec21-Jan22
assert test["dt"].between(pd.Timestamp("2021-11-30"), pd.Timestamp("2022-02-01")).all(), \
    "test datetimes parsed outside the expected window - check day/month order"
assert train["dt"].max() < test["dt"].min(), "train/test are not chronologically disjoint"
print("\nOK: parsing validated, train ends before test begins (pure forecasting split).")

## 1. EDA that tries to break our assumptions
Each block below is here because it can change what we build next, not to produce plots.

In [ ]:
# ---- EDA 1: target balance, missingness -----------------------------------
print("target balance")
print(train[TARGET].value_counts(normalize=True).round(4).to_string(), "\n")

print("missing values (train)")
print(train.isna().sum()[lambda s: s > 0].to_string())
print("\nmissing values (test)")
print(test.isna().sum()[lambda s: s > 0].to_string())

na_sensor_tr = train[S1].isna()
print(f"\nrows with BOTH sensors missing: train={int((train[S1].isna() & train[S2].isna()).sum())}, "
      f"test={int((test[S1].isna() & test[S2].isna()).sum())}")
print(f"fault rate | sensors missing : {train.loc[na_sensor_tr, TARGET].mean():.4f}")
print(f"fault rate | temp missing    : {train.loc[train['Temperature'].isna(), TARGET].mean():.4f}")
print(f"fault rate | overall         : {train[TARGET].mean():.4f}")
print("\nNOTE: test has ZERO rows with missing sensors, train has thousands. Those train rows")
print("carry no usable sensor signal and represent a regime the test set never contains.")

In [ ]:
# ---- EDA 2: sampling structure and the timestamp-resolution trap ----------
tr_sorted = train.sort_values("dt")
gaps = tr_sorted["dt"].diff().dt.total_seconds().dropna()
print("train inter-row gap (seconds)")
print(gaps.describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(2).to_string())

print(f"\nfraction of timestamps with seconds == 0:  train={(train['dt'].dt.second == 0).mean():.4f}"
      f"  test={(test['dt'].dt.second == 0).mean():.4f}")

tr_min = train["dt"].dt.floor("min")
te_min = test["dt"].dt.floor("min")
print(f"\nrows per distinct minute: train={len(train)/tr_min.nunique():.3f}  test={len(test)/te_min.nunique():.3f}")
print("rows-per-minute histogram (train):", tr_min.value_counts().value_counts().sort_index().to_dict())
print("rows-per-minute histogram (test) :", te_min.value_counts().value_counts().sort_index().to_dict())

print("""
=========================== CRITICAL FINDING ===========================
The TEST timestamps are truncated to the minute (every second == 0).
The TRAIN timestamps keep full second resolution.

Both files are the SAME underlying stream sampled ~2.18 rows per minute.

Any feature that depends on row ORDER (lags, neighbours, rolling windows)
can therefore be built more accurately on train than it ever can on test.
If we sort train by its exact second and test by minute, local validation
is measuring a precision we will not have at inference time.

=> Every notebook below floors BOTH tables to the minute and then breaks
   within-minute ties with the same arbitrary rule. Notebook B quantifies
   exactly how much this handicap costs.
========================================================================
""")

In [ ]:
# ---- EDA 3: are faults temporally clustered? ------------------------------
y_seq = tr_sorted[TARGET].values
change = np.flatnonzero(np.diff(y_seq)) + 1
run_lens = np.diff(np.concatenate([[0], change, [len(y_seq)]]))
run_lab = y_seq[np.concatenate([[0], change])]
runs = pd.DataFrame({"label": run_lab, "len": run_lens})
print(f"number of label runs: {len(runs):,} across {len(y_seq):,} rows")
print(runs.groupby("label")["len"].describe(percentiles=[.5, .9, .99]).round(2).to_string())

daily = train.assign(date=train["dt"].dt.date).groupby("date").agg(
    n=(TARGET, "size"), fault_rate=(TARGET, "mean"))
print(f"\ndaily fault rate: min={daily.fault_rate.min():.4f} max={daily.fault_rate.max():.4f} "
      f"std={daily.fault_rate.std():.5f} over {len(daily)} days")

hourly = train.assign(h=train["dt"].dt.hour).groupby("h")[TARGET].mean()
print(f"hourly fault rate: min={hourly.min():.4f} max={hourly.max():.4f} std={hourly.std():.5f}")

print("""
FINDING (contradicts the working plan):
Faults are NOT emitted in contiguous episodes. Median run length is 1 row
for the fault class. Faults are injected into individual, isolated rows.
The fault rate is also essentially constant across every day and every hour.

Two consequences:
  * "how long does a fault episode last" and "does fault frequency drift"
    are dead ends - there is nothing there to model.
  * BUT because a faulty row sits between healthy neighbours, its temporal
    neighbours give a clean estimate of the true ambient level at that
    moment. That is the exploitable structure.
""")

In [ ]:
# ---- EDA 4: hint 1, "you just need that line" -----------------------------
d = train.dropna(subset=[S1, S2]).copy()
d["diff"]  = d[S1] - d[S2]
d["ratio"] = d[S1] / (d[S2].abs() + EPS)
d["rel"]   = d["diff"] / (d[S2].abs() + EPS)

for lab in (0, 1):
    s = d[d[TARGET] == lab]
    slope, intercept = np.polyfit(s[S2], s[S1], 1)
    r = np.corrcoef(s[S2], s[S1])[0, 1]
    print(f"class {lab}: S1 = {slope:.4f} * S2 + {intercept:.4f}   (corr {r:.4f}, n={len(s):,})")

print()
print(d.groupby(TARGET)[["diff", "ratio", "rel"]]
        .describe(percentiles=[.01, .25, .5, .75, .99]).round(3).T.to_string())

# how far does the line alone get us?
best = (0.0, None, None)
for col in ("diff", "ratio", "rel"):
    for thr in np.quantile(d[col], np.linspace(0.02, 0.98, 300)):
        f = f1_score(d[TARGET], (d[col] >= thr).astype(int))
        if f > best[0]:
            best = (f, col, thr)
print(f"\nbest single-feature threshold rule: {best[1]} >= {best[2]:.4f}  ->  train F1 = {best[0]:.4f}")
print("""
The line is real: healthy rows sit on S1 ~= S2, faulty rows sit on a
visibly steeper line. But a threshold on it alone tops out well short of
the published 92.2% leaderboard baseline, so the line is a component of
the answer, not the answer.
""")

In [ ]:
# ---- EDA 5: the local-context signal --------------------------------------
# Rolling median over neighbouring rows estimates the ambient level.
# No labels are used, so this is computable identically on the test set.
probe = train.sort_values("dt").reset_index(drop=True)
for w in (7, 21, 121):
    for col, nm in ((S1, "S1"), (S2, "S2")):
        med = probe[col].rolling(w, center=True, min_periods=3).median()
        probe[f"{nm}_localratio_{w}"] = probe[col] / (med.abs() + EPS)

cols = [c for c in probe.columns if "localratio" in c]
summary = probe.groupby(TARGET)[cols].median().T
summary.columns = ["healthy", "faulty"]
summary["separation"] = (summary["healthy"] - summary["faulty"]).abs()
print("median of value / local-median, by class")
print(summary.round(4).to_string())

print("""
FINDING: a faulty row sits far BELOW its own temporal neighbourhood on both
sensors, and much further below on Sensor 2 than on Sensor 1. That is the
whole fault mechanism visible in one table, and it is exactly what hints 4
(lags) and 5 (aggregate) are pointing at. It also explains why the fault
class shows a steeper S1-vs-S2 line: S2 is pulled down harder than S1.
""")

In [ ]:
# ---- EDA 6: train/test drift and the "2022" hypothesis --------------------
def reldiff(df):
    return (df[S1] - df[S2]) / (df[S2].abs() + EPS)

train["_rel"] = reldiff(train)
test["_rel"]  = reldiff(test)

comp = pd.DataFrame({
    "train": train[[S1, S2, "Temperature", "Relative_Humidity", "_rel"]].median(),
    "test":  test[[S1, S2, "Temperature", "Relative_Humidity", "_rel"]].median(),
}).round(3)
print("median of each column, train vs test")
print(comp.to_string())

# A crude fault-share proxy that needs no labels: share of rows above a
# separating threshold on the relative difference.
THR_PROXY = 0.15
print(f"\nshare of rows with rel-diff > {THR_PROXY} (proxy for fault share)")
print(f"  train  : {(train['_rel'] > THR_PROXY).mean():.4f}   (true fault rate {train[TARGET].mean():.4f})")
by_month = test.assign(m=test["dt"].dt.to_period("M")).groupby("m")["_rel"].agg(
    n="size", proxy=lambda s: (s > THR_PROXY).mean()).round(4)
print(by_month.to_string())

print("""
FINDING on hint 3 ("I am 2022, I promise nothing wrong will happen"):
January 2022 shows the SAME fault-proxy share as December 2021 and as the
training period. There is no evidence that 2022 rows are disproportionately
healthy, and a hard rule mapping 2022 -> class 0 would destroy roughly half
the positive class on the private leaderboard.

The defensible reading is the boring one: train contains a single year, so
any raw year / month / day-of-year feature is a constant in training and an
unseen value at inference. The stage-by-stage runs below test this directly,
and calendar features do measurably HURT on future validation.

Temperature and relative humidity also drift (autumn -> winter), so they are
the covariates most likely to carry a spurious seasonal association. An
ablation below checks whether the model actually needs them.
""")

## 2. Feature construction
Families are defined separately so a stage can switch one on at a time.

In [ ]:
# ---- unified stream + feature builder -------------------------------------
# Features are built on train and test CONCATENATED, sorted by time. No label
# is ever consulted, no external data is used: this is only a way to give a
# test row the same kind of temporal neighbourhood a train row gets.

ORDER_SEED = 0          # controls arbitrary within-minute tie-breaking
K_NEIGHBOURS = 3        # neighbours used on each side (tuned in stage 5)

def build_stream(order_seed=ORDER_SEED):
    a = train.drop(columns=[c for c in ["_rel"] if c in train.columns]).copy()
    b = test.drop(columns=[c for c in ["_rel"] if c in test.columns]).copy()
    a["ID"] = -1
    a["is_test"] = 0
    b["is_test"] = 1
    b[TARGET] = np.nan
    full = pd.concat([a, b], ignore_index=True, sort=False)

    # Floor to the minute so train gets exactly the resolution test has.
    full["tmin"] = full["dt"].dt.floor("min")
    rng = np.random.RandomState(order_seed)
    full["_tie"] = rng.rand(len(full))
    full = full.sort_values(["tmin", "_tie"]).reset_index(drop=True)
    return full

def add_raw(f, full):
    f["S1"] = full[S1]
    f["S2"] = full[S2]
    f["Temperature"] = full["Temperature"]
    f["Relative_Humidity"] = full["Relative_Humidity"]
    return ["S1", "S2", "Temperature", "Relative_Humidity"]

def add_relation(f, full):
    diff = full[S1] - full[S2]
    f["diff"] = diff
    f["abs_diff"] = diff.abs()
    f["ratio"] = full[S1] / (full[S2].abs() + EPS)
    f["rel_diff"] = diff / (full[S2].abs() + EPS)
    f["sym_rel"] = 2 * diff / (full[S1].abs() + full[S2].abs() + EPS)
    f["log_ratio"] = np.log(full[S1].clip(lower=0.01)) - np.log(full[S2].clip(lower=0.01))
    f["level"] = (full[S1] + full[S2]) / 2
    f["resid_line"] = full[S1] - (1.0039 * full[S2] + 0.7551)   # healthy-class fit from EDA 4
    return ["diff", "abs_diff", "ratio", "rel_diff", "sym_rel", "log_ratio", "level", "resid_line"]

def add_calendar(f, full):
    f["hour"] = full["dt"].dt.hour
    f["dayofweek"] = full["dt"].dt.dayofweek
    f["day"] = full["dt"].dt.day
    f["minute_of_day"] = full["dt"].dt.hour * 60 + full["dt"].dt.minute
    f["hour_sin"] = np.sin(2 * np.pi * f["minute_of_day"] / 1440)
    f["hour_cos"] = np.cos(2 * np.pi * f["minute_of_day"] / 1440)
    return ["hour", "dayofweek", "day", "minute_of_day", "hour_sin", "hour_cos"]

def add_neighbour(f, full, k=K_NEIGHBOURS):
    """Lag/lead context. Faults are isolated rows, so neighbours estimate ambient."""
    names = []
    ratio = full[S1] / (full[S2].abs() + EPS)
    for col, nm in ((S1, "S1"), (S2, "S2"), (None, "ratio")):
        base = ratio if col is None else full[col]
        sides = {}
        for j in range(1, k + 1):
            sides[f"m{j}"] = base.shift(j)
            sides[f"p{j}"] = base.shift(-j)
            if col is not None:
                f[f"{nm}_lag{j}"] = sides[f"m{j}"]
                f[f"{nm}_lead{j}"] = sides[f"p{j}"]
                names += [f"{nm}_lag{j}", f"{nm}_lead{j}"]
        N = pd.DataFrame(sides)
        med, mean, sd = N.median(axis=1), N.mean(axis=1), N.std(axis=1)
        f[f"{nm}_nb_med"] = med
        f[f"{nm}_nb_std"] = sd
        f[f"{nm}_nb_dev"] = base - med
        f[f"{nm}_nb_ratio"] = base / (med.abs() + EPS)
        f[f"{nm}_nb_z"] = (base - mean) / (sd + EPS)
        names += [f"{nm}_nb_med", f"{nm}_nb_std", f"{nm}_nb_dev", f"{nm}_nb_ratio", f"{nm}_nb_z"]

    f["nb_ratio_gap"] = f["S1_nb_ratio"] / (f["S2_nb_ratio"].abs() + EPS)
    f["nb_dev_gap"] = f["S1_nb_dev"] - f["S2_nb_dev"]
    f["gap_prev_s"] = (full["tmin"] - full["tmin"].shift(1)).dt.total_seconds()
    f["gap_next_s"] = (full["tmin"].shift(-1) - full["tmin"]).dt.total_seconds()
    names += ["nb_ratio_gap", "nb_dev_gap", "gap_prev_s", "gap_next_s"]
    return names

def add_rolling(f, full, windows=("30min", "360min")):
    """Time-based windows: immune to within-minute ordering, unlike row windows."""
    names = []
    g = full.set_index("tmin")
    ratio_vals = (full[S1] / (full[S2].abs() + EPS)).values
    for w in windows:
        for col, nm in ((S1, "S1"), (S2, "S2")):
            med = g[col].rolling(w, center=True, closed="both").median().values
            f[f"{nm}_roll_ratio_{w}"] = full[col].values / (np.abs(med) + EPS)
            f[f"{nm}_roll_dev_{w}"] = full[col].values - med
            names += [f"{nm}_roll_ratio_{w}", f"{nm}_roll_dev_{w}"]
        rmed = (g[S1] / (g[S2].abs() + EPS)).rolling(w, center=True, closed="both").median().values
        f[f"ratio_roll_dev_{w}"] = ratio_vals - rmed
        names.append(f"ratio_roll_dev_{w}")
    return names

def build_features(order_seed=ORDER_SEED, k=K_NEIGHBOURS):
    full = build_stream(order_seed)
    f = pd.DataFrame(index=full.index)
    groups = {}
    groups["raw"] = add_raw(f, full)
    groups["relation"] = add_relation(f, full)
    groups["calendar"] = add_calendar(f, full)
    groups["neighbour"] = add_neighbour(f, full, k)
    groups["rolling"] = add_rolling(f, full)
    f = f.replace([np.inf, -np.inf], np.nan).astype(np.float32)
    return full, f, groups

t0 = time.time()
full, FEAT, GROUPS = build_features()
print(f"feature matrix {FEAT.shape} built in {time.time()-t0:.1f}s")
for g, cols in GROUPS.items():
    print(f"  {g:10s} {len(cols):3d} cols")

is_train = (full["is_test"] == 0).values
X_all = FEAT[is_train].reset_index(drop=True)
y_all = full.loc[is_train, TARGET].astype(int).values
dt_all = full.loc[is_train, "dt"].reset_index(drop=True)
X_test = FEAT[~is_train].reset_index(drop=True)
test_ids = full.loc[~is_train, "ID"].astype(int).values
print(f"\nX_all {X_all.shape} | X_test {X_test.shape} | positive rate {y_all.mean():.4f}")

STAGES = {
    "S0_raw":            GROUPS["raw"],
    "S1_relation":       GROUPS["raw"] + GROUPS["relation"],
    "S2_calendar":       GROUPS["raw"] + GROUPS["relation"] + GROUPS["calendar"],
    "S3_neighbour":      GROUPS["raw"] + GROUPS["relation"] + GROUPS["neighbour"],
    "S4_rolling":        GROUPS["raw"] + GROUPS["relation"] + GROUPS["neighbour"] + GROUPS["rolling"],
}

## 3. Validation framework

Model settings are frozen across stages 0-4 so that every score change is
attributable to the feature family, not to tuning.

In [ ]:
# ---- fixed model settings + threshold optimiser + metric logger -----------
CB_PARAMS = dict(
    iterations=600,
    learning_rate=0.08,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=SEED,
    thread_count=-1,
    verbose=0,
    allow_writing_files=False,
)
print("frozen CatBoost params for all feature stages:", CB_PARAMS)

def optimise_threshold(y_true, proba, n_grid=300):
    """Pick the probability cut that maximises F1 on the validation fold."""
    lo, hi = np.quantile(proba, 0.01), np.quantile(proba, 0.99)
    grid = np.unique(np.concatenate([np.linspace(lo, hi, n_grid), [0.5]]))
    scores = [f1_score(y_true, (proba >= t).astype(int)) for t in grid]
    i = int(np.argmax(scores))
    return float(grid[i]), float(scores[i])

def fold_report(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return dict(f1=f, precision=p, recall=r, threshold=threshold,
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                pos_rate=float(pred.mean()), auc=float(roc_auc_score(y_true, proba)))

METRIC_LOG = []

def log_result(row):
    METRIC_LOG.append(row)
    return row

def save_log(name):
    df = pd.DataFrame(METRIC_LOG)
    path = os.path.join(WORK, name)
    df.to_csv(path, index=False)
    print(f"saved {path}  ({len(df)} rows)")
    return df

def fit_predict(Xtr, ytr, Xva, params=None):
    m = CatBoostClassifier(**(params or CB_PARAMS))
    m.fit(Xtr, ytr)
    return m, m.predict_proba(Xva)[:, 1]

In [ ]:
# ---- validation A: GroupKFold on calendar date ----------------------------
# Rows from one calendar day never straddle the train/validation boundary,
# which removes most local temporal leakage while still giving several
# independent estimates and a full out-of-fold prediction vector.

N_FOLDS = 5
date_group = dt_all.dt.date.astype(str).values
print("distinct calendar dates:", len(np.unique(date_group)))

def run_date_cv(cols, stage_name, params=None, verbose=True):
    t0 = time.time()
    oof = np.full(len(X_all), np.nan)
    fold_rows = []
    gkf = GroupKFold(n_splits=N_FOLDS)
    for k, (itr, iva) in enumerate(gkf.split(X_all, y_all, date_group)):
        _, proba = fit_predict(X_all.loc[itr, cols], y_all[itr], X_all.loc[iva, cols], params)
        oof[iva] = proba
        thr, _ = optimise_threshold(y_all[iva], proba)
        rep = fold_report(y_all[iva], proba, thr)
        rep.update(fold=k, n_val=len(iva),
                   val_span=f"{dt_all[iva].min():%m-%d}..{dt_all[iva].max():%m-%d}")
        fold_rows.append(rep)
        if verbose:
            print(f"  fold {k}: F1={rep['f1']:.5f} P={rep['precision']:.4f} R={rep['recall']:.4f} "
                  f"thr={thr:.3f} posrate={rep['pos_rate']:.4f} dates {rep['val_span']}")

    folds = pd.DataFrame(fold_rows)
    pooled_thr, pooled_f1 = optimise_threshold(y_all, oof)
    pooled = fold_report(y_all, oof, pooled_thr)
    row = dict(stage=stage_name, n_features=len(cols),
               cv_f1_mean=folds.f1.mean(), cv_f1_std=folds.f1.std(),
               cv_f1_min=folds.f1.min(), cv_f1_max=folds.f1.max(),
               thr_mean=folds.threshold.mean(), thr_std=folds.threshold.std(),
               oof_f1=pooled["f1"], oof_thr=pooled_thr,
               oof_precision=pooled["precision"], oof_recall=pooled["recall"],
               oof_pos_rate=pooled["pos_rate"], oof_auc=pooled["auc"],
               f1_at_half=f1_score(y_all, (oof >= 0.5).astype(int)),
               tn=pooled["tn"], fp=pooled["fp"], fn=pooled["fn"], tp=pooled["tp"],
               runtime_s=round(time.time() - t0, 1))
    log_result(row)
    print(f"  -> {stage_name}: CV F1 {row['cv_f1_mean']:.5f} +/- {row['cv_f1_std']:.5f} | "
          f"OOF F1 {row['oof_f1']:.5f} @ thr {pooled_thr:.3f} | {row['runtime_s']}s")
    np.save(os.path.join(WORK, f"oof_{stage_name}.npy"), oof)
    return row, oof, folds

## 4. Staged experiments

In [ ]:
# ---- stages 0-4, model settings held constant -----------------------------
oof_store, fold_store = {}, {}
for stage_name, cols in STAGES.items():
    print(f"\n=== {stage_name}  ({len(cols)} features) ===")
    row, oof, folds = run_date_cv(cols, stage_name)
    oof_store[stage_name] = oof
    fold_store[stage_name] = folds

results = pd.DataFrame(METRIC_LOG)
print("\n" + "=" * 96)
print(results[["stage", "n_features", "cv_f1_mean", "cv_f1_std", "oof_f1",
               "oof_thr", "oof_precision", "oof_recall", "oof_pos_rate", "runtime_s"]]
      .round(5).to_string(index=False))

In [ ]:
# ---- what the stage deltas mean -------------------------------------------
base = results.set_index("stage")["oof_f1"]
print("stage-to-stage OOF F1 deltas")
for a, b in [("S0_raw", "S1_relation"), ("S1_relation", "S2_calendar"),
             ("S1_relation", "S3_neighbour"), ("S3_neighbour", "S4_rolling")]:
    if a in base and b in base:
        print(f"  {a:14s} -> {b:14s}  {base[b]-base[a]:+.5f}")
print("""
Read this carefully before spending a submission:
 * relation features give a small, real gain over raw sensors
 * calendar features are the only family that can move the score DOWN
 * neighbour features are the step change - this is the hidden feature
 * rolling adds little once neighbours are present (they overlap)
""")

## 5. Ablations and the neighbour-count sweep

In [ ]:
# ---- ablations: which columns actually earn their place -------------------
best_stage = "S4_rolling"
best_cols = STAGES[best_stage]

ablations = {
    "A_full":            best_cols,
    "A_no_weather":      [c for c in best_cols if c not in ("Temperature", "Relative_Humidity")],
    "A_no_raw_levels":   [c for c in best_cols if c not in ("S1", "S2", "level")],
    "A_neighbour_only":  GROUPS["raw"] + GROUPS["neighbour"],
    "A_plus_calendar":   best_cols + GROUPS["calendar"],
}
for name, cols in ablations.items():
    print(f"\n=== {name} ({len(cols)} features) ===")
    run_date_cv(cols, name, verbose=False)

In [ ]:
# ---- how many neighbours? -------------------------------------------------
# Rebuilding the stream is cheap; this is the one hyperparameter of the
# winning feature family, so it is worth measuring rather than guessing.
for k in (1, 2, 3, 5):
    full_k, feat_k, groups_k = build_features(k=k)
    m = (full_k["is_test"] == 0).values
    Xk = feat_k[m].reset_index(drop=True)
    yk = full_k.loc[m, TARGET].astype(int).values
    cols = groups_k["raw"] + groups_k["relation"] + groups_k["neighbour"] + groups_k["rolling"]
    gkf = GroupKFold(n_splits=N_FOLDS)
    grp = full_k.loc[m, "dt"].dt.date.astype(str).values
    scores = []
    for itr, iva in gkf.split(Xk, yk, grp):
        _, proba = fit_predict(Xk.loc[itr, cols], yk[itr], Xk.loc[iva, cols])
        thr, f = optimise_threshold(yk[iva], proba)
        scores.append(f)
    log_result(dict(stage=f"K_sweep_k{k}", n_features=len(cols),
                    cv_f1_mean=np.mean(scores), cv_f1_std=np.std(scores)))
    print(f"k={k}: CV F1 {np.mean(scores):.5f} +/- {np.std(scores):.5f}")
    del full_k, feat_k, Xk
    gc.collect()

## 6. Drift diagnostics

In [ ]:
# ---- adversarial validation: how different is test from train? ------------
adv_y = full["is_test"].values
adv_cols = STAGES["S4_rolling"]
idx = np.random.RandomState(SEED).permutation(len(FEAT))
cut = int(0.7 * len(idx))
adv = CatBoostClassifier(iterations=300, learning_rate=0.1, depth=5,
                         random_seed=SEED, verbose=0, thread_count=-1,
                         allow_writing_files=False)
adv.fit(FEAT.iloc[idx[:cut]][adv_cols], adv_y[idx[:cut]])
adv_p = adv.predict_proba(FEAT.iloc[idx[cut:]][adv_cols])[:, 1]
auc = roc_auc_score(adv_y[idx[cut:]], adv_p)
print(f"adversarial AUC (train vs test): {auc:.4f}")
imp = pd.Series(adv.get_feature_importance(), index=adv_cols).sort_values(ascending=False)
print("\ncolumns that separate train from test most strongly")
print(imp.head(10).round(2).to_string())
print("""
An AUC near 1.0 does NOT automatically mean trouble - the two files are
different calendar periods, so some separability is guaranteed. What matters
is WHICH columns do the separating. If Temperature and Relative_Humidity lead
the list, that is seasonal drift, and the no_weather ablation above tells us
whether the model leans on it. If the neighbour features led instead, we would
have a real problem.
""")

In [ ]:
# ---- feature importance on the chosen feature set -------------------------
m_full = CatBoostClassifier(**CB_PARAMS)
m_full.fit(X_all[best_cols], y_all)
imp = pd.Series(m_full.get_feature_importance(), index=best_cols).sort_values(ascending=False)
print(imp.head(25).round(3).to_string())
imp.to_csv(os.path.join(WORK, "feature_importance.csv"))

## 7. Light model refinement

In [ ]:
# ---- light model refinement (only after features are settled) -------------
tune_grid = [
    dict(depth=6, learning_rate=0.08, iterations=600),
    dict(depth=8, learning_rate=0.06, iterations=900),
    dict(depth=6, learning_rate=0.05, iterations=1200),
]
for i, g in enumerate(tune_grid):
    p = dict(CB_PARAMS); p.update(g)
    print(f"\n=== tune_{i} {g} ===")
    run_date_cv(best_cols, f"tune_{i}", params=p, verbose=False)

log_df = save_log("notebookA_metrics.csv")
with open(os.path.join(WORK, "notebookA_summary.json"), "w") as fh:
    json.dump({r["stage"]: {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                            for k, v in r.items() if k != "stage"} for r in METRIC_LOG}, fh, indent=2)
print(f"\ntotal notebook runtime so far: {(time.time()-t_notebook_start)/60:.1f} min")

## 8. Final models on all training data + candidate submissions
Files are written to `/kaggle/working`. Nothing is submitted.

In [ ]:
# ---- final models on ALL training data + candidate submissions ------------
def make_submission(cols, threshold, tag, params=None, n_seeds=1):
    t0 = time.time()
    probs = np.zeros(len(X_test))
    for s in range(n_seeds):
        p = dict(params or CB_PARAMS)
        p["random_seed"] = SEED + s
        m = CatBoostClassifier(**p)
        m.fit(X_all[cols], y_all)
        probs += m.predict_proba(X_test[cols])[:, 1]
    probs /= n_seeds
    pred = (probs >= threshold).astype(int)
    sub = pd.DataFrame({"ID": test_ids, TARGET: pred}).sort_values("ID").reset_index(drop=True)
    if sample is not None:
        assert len(sub) == len(sample), f"row count {len(sub)} != sample {len(sample)}"
        assert set(sub["ID"]) == set(sample["ID"]), "ID mismatch against sample_submission"
        sub = sub.set_index("ID").loc[sample["ID"]].reset_index()
    path = os.path.join(WORK, f"submission_{tag}.csv")
    sub.to_csv(path, index=False)
    prob_path = os.path.join(WORK, f"test_proba_{tag}.npy")
    np.save(prob_path, probs)
    print(f"{tag:26s} thr={threshold:.4f}  predicted positive rate={pred.mean():.4f}  "
          f"({time.time()-t0:.0f}s)  -> {os.path.basename(path)}")
    return sub, probs

In [ ]:
# ---- candidate submissions (NOT submitted automatically) ------------------
final = pd.DataFrame(METRIC_LOG)
best_row = final.loc[final["oof_f1"].idxmax()]
print("best configuration by OOF F1:")
print(best_row[["stage", "n_features", "cv_f1_mean", "cv_f1_std", "oof_f1", "oof_thr"]].to_string())

thr_best = float(best_row["oof_thr"])
cand = {}
cand["A_neighbour_oofthr"] = make_submission(STAGES["S3_neighbour"],
                                             float(final.set_index("stage").loc["S3_neighbour", "oof_thr"]),
                                             "A_neighbour_oofthr")
cand["A_full_oofthr"] = make_submission(best_cols, thr_best, "A_full_oofthr", n_seeds=3)
cand["A_full_half"] = make_submission(best_cols, 0.5, "A_full_thr050", n_seeds=3)

print("\nagreement between candidates")
keys = list(cand)
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        a = cand[keys[i]][0][TARGET].values
        b = cand[keys[j]][0][TARGET].values
        print(f"  {keys[i]:22s} vs {keys[j]:22s}: {(a == b).mean():.5f} identical")

print(f"\nfiles in {WORK}:")
for f_ in sorted(os.listdir(WORK)):
    print("  ", f_)
print(f"\ntotal runtime: {(time.time()-t_notebook_start)/60:.1f} min")
print("\nNothing has been submitted. Review these numbers against notebook B first.")

## What to bring back

* `notebookA_metrics.csv` - every stage, fold spread, thresholds, confusion counts
* `feature_importance.csv`
* `oof_*.npy` - out-of-fold probabilities for blending or threshold analysis later
* `submission_A_*.csv` - candidates only

The two numbers that matter most when comparing with notebook B: the **fold-to-fold standard deviation** and whether the **stage ordering** is the same under both validation schemes. If the ordering agrees, the feature conclusions are safe. If a stage looks good here and bad under forward validation, trust the forward one, because the real test set is the future.